# AgriEdge - Tomato Leaf Disease Classifier (Training)

**Person 1 / Edge AI.** Trains a 3-class tomato leaf disease classifier
(Healthy, Early Blight, Late Blight) via MobileNetV2 transfer learning,
evaluates it honestly on an untouched test set, and exports a
TensorFlow Lite model for local (laptop) inference.

**Before running:** `Runtime -> Change runtime type -> select a GPU (e.g. T4)`.
Training on CPU will still work but is much slower.

Run cells top to bottom. Every number in this notebook is computed from
the actual downloaded dataset and the actual trained model -- nothing
here is a placeholder.

In [5]:
import random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

import os
os.makedirs("results", exist_ok=True)
os.makedirs("export", exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU") or "None (will train on CPU - slower)")

TensorFlow: 2.21.0
GPU available: None (will train on CPU - slower)


## 1. Get the dataset

We use the **PlantVillage** tomato subset from `spMohanty/PlantVillage-Dataset`
on GitHub. Rather than cloning the whole repo (14 crops, tens of
thousands of images), we `svn export` only the 3 folders we need --
much faster for a hackathon.

**If `svn` fails in your Colab session** (rare), use the fallback cell
below it instead: download the "PlantVillage Dataset" from Kaggle
manually, upload the zip to Colab (folder icon on the left -> upload),
then `!unzip` it into `PlantVillage-Dataset/raw/color/`.

In [4]:
!apt-get -qq install subversion > /dev/null 2>&1

DATASET_DIR = "PlantVillage-Dataset/raw/color"
os.makedirs(DATASET_DIR, exist_ok=True)

RAW_CLASS_FOLDERS = ["Tomato___healthy", "Tomato___Early_blight", "Tomato___Late_blight"]
BASE_URL = "https://github.com/spMohanty/PlantVillage-Dataset/trunk/raw/color"

for folder in RAW_CLASS_FOLDERS:
    target = f"{DATASET_DIR}/{folder}"
    if os.path.isdir(target) and len(os.listdir(target)) > 0:
        print(f"Already present: {folder} ({len(os.listdir(target))} files)")
        continue
    print(f"Downloading {folder} ...")
    !svn export --force "{BASE_URL}/{folder}" "{target}"

print("Done. Folders present:", os.listdir(DATASET_DIR))

The system cannot find the path specified.


'svn' is not recognized as an internal or external command,
operable program or batch file.


'svn' is not recognized as an internal or external command,
operable program or batch file.


Done. Folders present: []


'svn' is not recognized as an internal or external command,
operable program or batch file.


**Fallback (only if the cell above failed):**
1. Go to kaggle.com, search "PlantVillage Dataset", download it (needs a free Kaggle account).
2. In Colab's left sidebar, click the folder icon -> upload the zip.
3. Run: `!unzip -q your-file.zip -d PlantVillage-Dataset-manual` then adjust
   `RAW_TO_CLEAN` paths in the next cell to match wherever the tomato
   folders ended up.

## 2. Verify the dataset before touching a model

This is not optional. We confirm the exact folders exist, then count,
balance-check, corruption-check, and visually inspect the data -- all
computed from what's actually on disk.

In [ ]:
# CLASS_NAMES must match ai/inference/config.py exactly -- this order
# becomes the model's output order, saved to model_metadata.json later
# so the rest of the app always stays in sync with what was trained.
CLASS_NAMES = ["Healthy", "Early Blight", "Late Blight"]

RAW_TO_CLEAN = {
    "Tomato___healthy": "Healthy",
    "Tomato___Early_blight": "Early Blight",
    "Tomato___Late_blight": "Late Blight",
}

class_dirs = {}
for raw_name, clean_name in RAW_TO_CLEAN.items():
    path = os.path.join(DATASET_DIR, raw_name)
    assert os.path.isdir(path), f"Missing expected folder: {path}"
    class_dirs[clean_name] = path

print("All required class folders found:")
for clean_name, path in class_dirs.items():
    print(f"  {clean_name:15s} -> {path}")

In [ ]:
# Actual counts, straight from disk -- never assumed.
image_counts = {name: len(os.listdir(path)) for name, path in class_dirs.items()}

print("Image counts per class:")
for name, count in image_counts.items():
    print(f"  {name:15s} {count}")
print(f"  {'TOTAL':15s} {sum(image_counts.values())}")

counts = list(image_counts.values())
imbalance_ratio = max(counts) / min(counts)
print(f"\nClass balance ratio (largest/smallest): {imbalance_ratio:.2f}")
print("Meaningful imbalance -- will use class weights." if imbalance_ratio > 1.5 else "Reasonably balanced.")

In [ ]:
from PIL import Image, UnidentifiedImageError

all_files, labels = [], []
for clean_name, dir_path in class_dirs.items():
    for fname in os.listdir(dir_path):
        all_files.append(os.path.join(dir_path, fname))
        labels.append(clean_name)

print(f"Checking all {len(all_files)} images for corruption (this reads every file once)...")
corrupted = []
for p in all_files:
    try:
        with Image.open(p) as img:
            img.verify()
    except (UnidentifiedImageError, OSError):
        corrupted.append(p)

print(f"Corrupted/unreadable images: {len(corrupted)}")
if corrupted:
    for p in corrupted[:10]:
        print("  ", p)
    corrupted_set = set(corrupted)
    kept = [(f, l) for f, l in zip(all_files, labels) if f not in corrupted_set]
    all_files, labels = [f for f, l in kept], [l for f, l in kept]
print(f"Proceeding with {len(all_files)} verified-readable images.")

In [ ]:
import matplotlib.pyplot as plt

random.seed(SEED)
fig, axes = plt.subplots(len(class_dirs), 4, figsize=(14, 3.2 * len(class_dirs)))
for row, (clean_name, dir_path) in enumerate(class_dirs.items()):
    files_here = os.listdir(dir_path)
    samples = random.sample(files_here, min(4, len(files_here)))
    for col, fname in enumerate(samples):
        img = Image.open(os.path.join(dir_path, fname))
        ax = axes[row, col]
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(clean_name if col == 0 else "", fontsize=10, loc="left")
plt.suptitle("Sample images per class (from the actual downloaded dataset)")
plt.tight_layout()
plt.savefig("results/sample_images.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
sample_files = random.sample(all_files, min(30, len(all_files)))
dims = [Image.open(f).size for f in sample_files]
widths, heights = zip(*dims)
print(f"Sampled {len(sample_files)} images for dimension inspection.")
print(f"Width  - min:{min(widths)} max:{max(widths)} mean:{sum(widths)/len(widths):.0f}")
print(f"Height - min:{min(heights)} max:{max(heights)} mean:{sum(heights)/len(heights):.0f}")

## 3. Train / validation / test split (70 / 15 / 15)

PlantVillage doesn't ship an official split, so we create one ourselves:
stratified by class, fixed seed, and split over distinct files so the
same image never appears in two subsets. The test set is never touched
again until final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files, labels, test_size=0.30, stratify=labels, random_state=SEED
)
val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED
)

print(f"train: {len(train_files)}   val: {len(val_files)}   test: {len(test_files)}")
for split_name, split_labels in [("train", train_labels), ("val", val_labels), ("test", test_labels)]:
    print(f"  {split_name}: {dict(Counter(split_labels))}")

## 4. Preprocessing + augmentation

Every image is resized to 224x224 and scaled to [-1, 1] (MobileNetV2's
expected input range) -- the *same* math `ai/inference/preprocess.py`
uses for local inference, so there's no train/inference mismatch.
Augmentation (flip / small rotation / zoom / shift / brightness) is
applied to the **training set only** -- validation and test images are
never randomly altered.

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

label_to_index = {name: i for i, name in enumerate(CLASS_NAMES)}

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.RandomBrightness(0.1),
], name="data_augmentation")


def make_dataset(files, str_labels, training):
    indices = [label_to_index[l] for l in str_labels]
    ds = tf.data.Dataset.from_tensor_slices((files, indices))

    def _load(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE)
        return image, label

    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=len(files), seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    # Official TF function here (full TF is available in Colab); it's the
    # same x/127.5 - 1 formula ai/inference/preprocess.py reimplements.
    ds = ds.map(
        lambda x, y: (tf.keras.applications.mobilenet_v2.preprocess_input(x), y),
        num_parallel_calls=AUTOTUNE,
    )
    return ds.prefetch(AUTOTUNE)


train_ds = make_dataset(train_files, train_labels, training=True)
val_ds = make_dataset(val_files, val_labels, training=False)
test_ds = make_dataset(test_files, test_labels, training=False)

for images, lbls in train_ds.take(1):
    print("Example batch:", images.shape, lbls.shape)

## 5. Class weights

If the classes are imbalanced, weight the loss so the model does not just learn to predict the majority class.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_indices = [label_to_index[l] for l in train_labels]
class_weight_values = compute_class_weight(
    class_weight="balanced", classes=np.arange(len(CLASS_NAMES)), y=train_indices
)
class_weight_dict = dict(enumerate(class_weight_values))
print("Class weights (from actual training-split counts):", class_weight_dict)

## 6. Model: MobileNetV2 transfer learning

MobileNetV2 is a small, fast convolutional network built for mobile/edge
devices -- a good fit since the final model needs to run locally rather
than on a server. We reuse its ImageNet-pretrained features (transfer
learning) instead of training a network from scratch, which needs far
less data and time.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,), include_top=False, weights="imagenet"
)
base_model.trainable = False  # freeze for head training

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
x = base_model(inputs, training=False)  # keep BatchNorm in inference mode
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="agriedge_tomato_classifier")
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("best_head_model.keras", monitor="val_accuracy", save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

## 7. Train the classification head (base frozen)

In [ ]:
HEAD_EPOCHS = 15

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

## 8. Optional fine-tuning

Now unfreeze the top ~30 layers of MobileNetV2 and continue training
with a much lower learning rate, so the pretrained features adapt
slightly to tomato leaves without being destroyed. BatchNorm layers
stay frozen even among the unfrozen layers (TensorFlow's own
transfer-learning guidance) since updating their running statistics
tends to destabilize fine-tuning on a small dataset.

In [ ]:
base_model.trainable = True

FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False
for layer in base_model.layers[FINE_TUNE_AT:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

FINE_TUNE_EPOCHS = 10
total_epochs = HEAD_EPOCHS + FINE_TUNE_EPOCHS

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history_head.epoch[-1] + 1,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

## 9. Training curves

In [ ]:
def combine(hist_a, hist_b, key):
    return hist_a.history.get(key, []) + hist_b.history.get(key, [])

acc = combine(history_head, history_fine, "accuracy")
val_acc = combine(history_head, history_fine, "val_accuracy")
loss = combine(history_head, history_fine, "loss")
val_loss = combine(history_head, history_fine, "val_loss")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(acc, label="train"); axes[0].plot(val_acc, label="val")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(loss, label="train"); axes[1].plot(val_loss, label="val")
axes[1].set_title("Loss"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## 10. Evaluate on the untouched test set

This is the only place test data is used. Accuracy alone can hide a
model that just always predicts the majority class, so we report
precision/recall/F1 per class too.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for images, lbls in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(lbls.numpy())

print("=== Test set evaluation ===")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

test_accuracy = float(np.mean(np.array(y_true) == np.array(y_pred)))
print(f"Test accuracy (Keras model): {test_accuracy:.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion Matrix (test set)")
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig("results/confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Error analysis -- useful regardless of how good the numbers look.
y_true_arr, y_pred_arr = np.array(y_true), np.array(y_pred)
wrong_idx = np.where(y_true_arr != y_pred_arr)[0]
print(f"{len(wrong_idx)} of {len(y_true_arr)} test images misclassified.")

if len(wrong_idx) > 0:
    show_idx = wrong_idx[:8]
    fig, axes = plt.subplots(1, len(show_idx), figsize=(3 * len(show_idx), 3.2))
    if len(show_idx) == 1:
        axes = [axes]
    for ax, idx in zip(axes, show_idx):
        ax.imshow(Image.open(test_files[idx]))
        ax.axis("off")
        ax.set_title(f"true: {CLASS_NAMES[y_true_arr[idx]]}\npred: {CLASS_NAMES[y_pred_arr[idx]]}", fontsize=9)
    plt.tight_layout()
    plt.savefig("results/misclassified_examples.png", dpi=120, bbox_inches="tight")
    plt.show()

## 11. Convert to TensorFlow Lite

We apply **dynamic range quantization**: weights are stored at lower
precision, giving a noticeably smaller, faster model with typically a
very small accuracy cost. We verify that tradeoff below instead of
assuming it.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = "export/tomato_disease_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

keras_path = "export/tomato_disease_model.keras"
model.save(keras_path)

keras_size_mb = os.path.getsize(keras_path) / 1e6
tflite_size_mb = os.path.getsize(tflite_path) / 1e6
print(f"Keras model size:  {keras_size_mb:.2f} MB")
print(f"TFLite model size: {tflite_size_mb:.2f} MB")

## 12. Verify the exported TFLite model actually works

We load the real `.tflite` file and run it on the real test set -- not assumed, checked.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite input: ", input_details[0]["shape"], input_details[0]["dtype"])
print("TFLite output:", output_details[0]["shape"], output_details[0]["dtype"])

# Uses the exact same preprocessing formula as ai/inference/preprocess.py
tflite_correct = 0
for i, path in enumerate(test_files):
    img = Image.open(path).convert("RGB").resize(IMAGE_SIZE)
    arr = np.asarray(img, dtype=np.float32)
    arr = (arr / 127.5) - 1.0
    arr = np.expand_dims(arr, axis=0)

    interpreter.set_tensor(input_details[0]["index"], arr)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]["index"])[0]
    if int(np.argmax(output)) == y_true[i]:
        tflite_correct += 1

tflite_accuracy = tflite_correct / len(test_files)
print(f"Keras test accuracy:  {test_accuracy:.4f}")
print(f"TFLite test accuracy: {tflite_accuracy:.4f}")
print(f"Difference: {abs(test_accuracy - tflite_accuracy):.4f}")

## 13. Save model metadata

`ai/inference/config.py` reads this file automatically, so class order/version always stays in sync with whatever was actually trained.

In [ ]:
import json
import datetime

metadata = {
    "model_version": "v1",
    "crop": "Tomato",
    "class_names": CLASS_NAMES,
    "input_size": list(IMAGE_SIZE),
    "created_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "test_accuracy": round(tflite_accuracy, 4),
    "test_set_size": len(test_files),
    "confidence_threshold": 0.80,
    "notes": "MobileNetV2 transfer learning, PlantVillage tomato subset, dynamic-range quantized TFLite export.",
}

with open("export/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

## 14. Download your results

Grab these two files and place them in `ai/models/` in your repo
(overwriting the empty placeholders):
- `export/tomato_disease_model.tflite`
- `export/model_metadata.json`

The cell below also prints a ready-to-paste block for the README's
"Actual Results" section.

In [ ]:
from google.colab import files

files.download("export/tomato_disease_model.tflite")
files.download("export/model_metadata.json")

print("=" * 70)
print("COPY-PASTE THIS INTO README.md 'Actual Results' SECTION:")
print("=" * 70)
summary_lines = [
    f"- Test accuracy (TFLite model): {tflite_accuracy:.1%}",
    f"- Test set size: {len(test_files)} images (of {len(all_files)} total, 70/15/15 split)",
    "- Per-class precision/recall/F1: see classification_report output in Section 10",
    f"- Model size: {tflite_size_mb:.2f} MB (TFLite, dynamic-range quantized, from {keras_size_mb:.2f} MB Keras)",
]
print("\n".join(summary_lines))